In [ ]:
import time, json
import pandas as pd
from datetime import datetime
from pathlib import Path
from utils import Helper

utils = Helper()
json_dir = Path(r"gstn_json")
final_content = []
for file_path in json_dir.glob("*.json"):
    print(str(file_path))
    data = utils.load_json(str(file_path))
    status = data.get("filingStatus",[])
    for items in status:
        for item in items:
            item["PAN_NO"] = file_path.stem
    
        final_content.extend(items)

pd.DataFrame(final_content).to_excel("PAN_DATA.xlsx", index=False)

In [1]:
import undetected_chromedriver as uc
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
import time
import pandas as pd
from datetime import datetime

cpath = r"C:\Users\kaustubh.keny\Downloads\Updated Company List (Listed  Unlisted)_Latest CFD.xlsx"
df = pd.read_excel(cpath, sheet_name = "All")
df.head(5)

,Status,Workstation Name,New cogencis Name (The & Ltd.),ISIN No,Company ID,PAN No
0,Listed,Infosys Ltd,Infosys Ltd.,INE009A01021,3486,AAACI4798L
1,Listed,GM Breweries Ltd,GM Breweries Ltd.,INE075D01018,2765,AAACG1653N
2,Listed,Havells India Ltd,Havells India Ltd.,INE176B01034,3060,AAACH0351E
3,Listed,Vimta Labs Ltd,Vimta Labs Ltd.,INE579C01029,8213,AAACV7244E
4,Listed,Swarna Securities Ltd,Swarna Securities Ltd.,INE595G01018,7452,AADCS1718H


In [8]:
df.columns

Index(['Status', 'Workstation Name', 'New cogencis Name (The & Ltd.)',
       'ISIN No', 'Company ID', 'GST No'],
      dtype='object')

In [7]:
companies = df[["Workstation Name","PAN No","Status"]]
condition1 = companies["PAN No"].apply(lambda x: len(str(x))>2)

df = companies[condition1]

print(len(df))


df.head(10)

7494


,Workstation Name,PAN No,Status
0,Infosys Ltd,AAACI4798L,Listed
1,GM Breweries Ltd,AAACG1653N,Listed
2,Havells India Ltd,AAACH0351E,Listed
3,Vimta Labs Ltd,AAACV7244E,Listed
4,Swarna Securities Ltd,AADCS1718H,Listed
5,Kretto Syscon Ltd,AAACI4341M,Listed
6,Tata Teleservices (Maharashtra) Ltd,AAACH1458C,Listed
7,Tata Consultancy Services Ltd,AAACR4849R,Listed
8,Transformers and Rectifiers (India) Ltd,AACCT8243P,Listed
9,L&T Finance Ltd,AACCA1963B,Listed


In [8]:
import pandas as pd

# Assuming df is your DataFrame
listed_df = df[df['Status'].str.lower() == 'listed']
unlisted_df = df[df['Status'].str.lower() == 'unlisted']

listed_df.to_json('listed.json', orient='records', indent=4)
unlisted_df.to_json('unlisted.json', orient='records', indent=4)

print("JSON files created successfully.")

JSON files created successfully.


In [10]:
# driver = uc.Chrome(version_main=145)
driver = webdriver.Chrome()
wait = WebDriverWait(driver, 20)

BASE_URL = "https://www.knowyourgst.com/gst-number-search/by-name-pan/"

driver.get(BASE_URL)
results = []
for idx, company in enumerate(companies[:]):
    # Find the input field by id
    
    input_box = driver.find_element(By.ID, "gstnumber")
    input_box.clear()
    input_box.send_keys(company)

    submit_btn = driver.find_element(By.CSS_SELECTOR, "input[type='submit']")
    submit_btn.click()
    
    time.sleep(1)
    boxes = driver.find_elements(By.CSS_SELECTOR, "div#searchresult")
    print(f"{idx + 1} RUNNING FOR: {company} Total: {len(boxes)}")
    
    if not boxes:
        results.append({
            "Query":company,
            "Company": "",
            "State": "",
            "GSTIN": "",
            "Link": ""
        })
    
    for box in boxes:

        link = box.find_element(By.TAG_NAME, "a").get_attribute("href")
        company_name = box.find_element(By.TAG_NAME, "h5").text.strip()
        strongs = box.find_elements(By.CSS_SELECTOR, "span.black-text strong")
        state = strongs[0].text.strip() if len(strongs) > 0 else ""
        gst_number = strongs[1].text.strip() if len(strongs) > 1 else ""

        results.append({
            "Query":company,
            "Company": company_name,
            "State": state,
            "GSTIN": gst_number,
            "Link": link
        })

driver.quit()

csv_name = f"gst_results_{datetime.now().strftime("%H%M%S")}.csv"

df = pd.DataFrame(results)
df.to_csv(csv_name, index=False)
print(f"[SAVED] {csv_name}")

#1618 run

1 RUNNING FOR: Blue Yonder India Pvt Ltd Total: 0
2 RUNNING FOR: Paragon Polymer Products Pvt Ltd Total: 3
3 RUNNING FOR: Emerson Electric Company (India) Pvt Ltd Total: 2
4 RUNNING FOR: Tamil Nadu State Transport Corporation Coimbatore Ltd Total: 1
5 RUNNING FOR: Dhanraj Solvex Pvt Ltd Total: 0
6 RUNNING FOR: Voltbek Home Appliances Pvt Ltd Total: 0
7 RUNNING FOR: Kanti Bijlee Utpadan Nigam Ltd Total: 0
8 RUNNING FOR: Gimatex Industries Pvt Ltd Total: 2
9 RUNNING FOR: Citiustech Healthcare Technology Pvt Ltd Total: 0
10 RUNNING FOR: Roma Builders Pvt Ltd Total: 1
11 RUNNING FOR: Daeseong India Automotive Pvt Ltd Total: 0
12 RUNNING FOR: Cmr-Toyotsu Aluminium India Pvt Ltd Total: 0
13 RUNNING FOR: Roche Diagnostics India Pvt Ltd Total: 2
14 RUNNING FOR: V-Trans India Ltd Total: 0
15 RUNNING FOR: Kellogg India Pvt Ltd Total: 9
16 RUNNING FOR: Emerson Process Management (India) Pvt Ltd Total: 9
17 RUNNING FOR: Milacron India Pvt Ltd Total: 0
18 RUNNING FOR: Sandhya Marines Ltd Total: 0
1

In [4]:
from datetime import datetime
csv_name = f"gst_results_{datetime.now().strftime("%H%M%S")}.csv"

df = pd.DataFrame(results)
df.to_csv(csv_name, index=False)
print(f"[SAVED] {csv_name}")


[SAVED] gst_results_103904.csv
